A notebook to compute average "partisan bias" scores by state & chamber

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from knobs_functions import *
import warnings

warnings.filterwarnings('ignore')


Calculate the average value of "partisan bias" metrics by state & chamber.
Also calculate where in the range of plans the unbiased plan falls. 

In [ ]:
from typing import List, Dict, Tuple, Any
from fetch import _score_mapping

metrics: List[str] = ["disproportionality", "efficiency_gap", "geometric_seats_bias", "seats_bias", "votes_bias", "mean_median_average_district", "lopsided_outcomes", "declination"]
additions: Dict[str, str] = dict(zip(metrics, metrics))
_score_mapping.update(additions)

ensembles = ["base0", "pop_minus", "pop_plus", "distpair", "ust", "distpair_ust", "reversible", "county25", "county50", "county75", "county100"]
table_list = [[x] for x in ensembles]

bias_tables: Dict[str, Dict[Tuple[str, str], Any]] = dict()
zero_tables: Dict[str, Dict[Tuple[str, str], Any]] = dict()

for variants in table_list:
    table: Dict[Tuple[str, str], Any] = dict()
    zero_table: Dict[Tuple[str, str], Any] = dict()

    for state, chamber in state_chamber_list:
        table[(state, chamber)] = dict()
        zero_table[(state, chamber)] = dict()

        for m in metrics:
            all_values: List[float] = list()
            for e in variants:
                arr = fetch_score_array(state, chamber, e, m)
                all_values.extend(arr)
            # Guard for undefined declinations
            all_values = [x for x in all_values if not np.isnan(x)]

            mean_value = np.mean(all_values)
            table[(state, chamber)][m] = mean_value

            zero_value = sum(1 for x in all_values if x < 0) / len(all_values)
            zero_table[(state, chamber)][m] = zero_value

    bias_tables[variants[0]] = table
    zero_tables[variants[0]] = zero_table


Generate interim tables so the above doesn't have to be re-run every time.

In [ ]:
print("{")
for variant in bias_tables:
    print(f'  "{variant}": {{')
    for (state, chamber) in bias_tables[variant]:
        print(f'    "{state}_{chamber}": {{')
        for m in bias_tables[variant][(state, chamber)]:
            formatted_value: str = f'{bias_tables[variant][(state, chamber)][m]:.10f}'
            print(f'      "{m}": {formatted_value},')
        print("    },")
    print("  },")

print("}")

In [ ]:
print("{")
for variant in zero_tables:
    print(f'  "{variant}": {{')
    for (state, chamber) in zero_tables[variant]:
        print(f'    "{state}_{chamber}": {{')
        for m in zero_tables[variant][(state, chamber)]:
            formatted_value: str = f'{zero_tables[variant][(state, chamber)][m]:.10f}'
            print(f'      "{m}": {formatted_value},')
        print("    },")
    print("  },")

print("}")

Convert the dict to a pandas DataFrame and LaTex

In [2]:
from typing import List, Dict, Tuple, Any

def make_table(table: Dict[Tuple[str, str], Any], *, 
                             latex_filename=None, 
                             markdown_filename=None,
                             values: bool = True,
                             rounding: int = 2):

    index_list = [f'{a[0]} {a[1]}' for a in state_chamber_list]
    df = pd.DataFrame(columns=metrics, index=index_list)

    for state, chamber in state_chamber_list:
        for m in metrics:
            multiplier = 1 if (m == "declination" and values) else 100
            df.loc[f'{state} {chamber}', m] = table[(state, chamber)][m] * multiplier
    df = df.applymap(pd.to_numeric)
    df = df.round(rounding)
    
    # Prepare state/chamber labels with seat counts
    state_chamber_size_dict = {f'{state} {chamber}': f'{state} {num_seats_dict[(state, chamber)]}' 
                              for state, chamber in state_chamber_list}
    
    # Greek letters for different formats
    greek_latex = {
        'alpha': 'α',
        'beta': 'β', 
        'delta': 'δ'
    }
    
    greek_unicode = {
        'alpha': 'α',
        'beta': 'β', 
        'delta': 'δ'
    }
    
    # Metric name mappings
    metrics_name_dict_latex = {
        "disproportionality": "PR", 
        "efficiency_gap": "EG",
        "geometric_seats_bias": f"${greek_latex['beta']}$",
        "seats_bias": f"${greek_latex['alpha']}_s$",
        "votes_bias": f"${greek_latex['alpha']}_v$",
        "mean_median_average_district": "mM", 
        "lopsided_outcomes": "LO", 
        "declination": greek_latex['delta']
    }
    
    metrics_name_dict_markdown = {
        "disproportionality": "PR", 
        "efficiency_gap": "EG",
        "geometric_seats_bias": greek_unicode['beta'],
        "seats_bias": greek_unicode['alpha'] + "ₛ",  # Unicode subscript
        "votes_bias": greek_unicode['alpha'] + "ᵥ",  # Unicode subscript
        "mean_median_average_district": "mM", 
        "lopsided_outcomes": "LO", 
        "declination": greek_unicode['delta']
    }
    
    # Generate LaTeX table
    if latex_filename is not None:
        df_latex = df.copy()
        df_latex = df_latex.applymap(lambda x: f"{x:.2f}")
        
        # Add LaTeX color formatting
        for state, chamber in state_chamber_list:
            for m in metrics:
                val = df.loc[f'{state} {chamber}', m]
                df_latex.loc[f'{state} {chamber}', m] = f'\\textcolor{{black}}{{ {val:.2f} }}'
        
        df_latex.rename(columns=metrics_name_dict_latex, index=state_chamber_size_dict, inplace=True)
        df_latex.to_latex(latex_filename, escape=False, column_format='l' + 'r' * len(df_latex.columns))
        # df_latex.to_latex(latex_filename, escape=False)
    
    # Generate Markdown table
    if markdown_filename is not None:
        df_markdown = df.copy()
        df_markdown = df_markdown.applymap(lambda x: f"{x:.2f}")
        df_markdown.rename(columns=metrics_name_dict_markdown, index=state_chamber_size_dict, inplace=True)
        
        # Create markdown table manually for better control
        with open(markdown_filename, 'w', encoding='utf-8') as f:
            # Write header
            headers = ['District'] + list(df_markdown.columns)
            f.write('| ' + ' | '.join(headers) + ' |\n')
            f.write('|' + '|'.join(['-' * (len(h) + 2) for h in headers]) + '|\n')
            
            # Write data rows
            for idx in df_markdown.index:
                row = [idx] + [str(df_markdown.loc[idx, col]) for col in df_markdown.columns]
                f.write('| ' + ' | '.join(row) + ' |\n')

    return df

Note: Switch the output location for the LaTeX for the A0 table

In [11]:
for variant in bias_tables:
    table = bias_tables[variant]
    latex_out: str = f'temp/partisan_bias_table_{variant}.tex'
    markdown_out: str = f'temp/partisan_bias_table_{variant}.md'
    make_table(table, latex_filename=latex_out, markdown_filename=markdown_out)

    table = zero_tables[variant]
    latex_out: str = f'temp/partisan_zero_table_{variant}.tex'
    markdown_out: str = f'temp/partisan_zero_table_{variant}.md'
    make_table(table, latex_filename=latex_out, markdown_filename=markdown_out, values=False)

Generate the "zero" or "unbiased" placement table for the all up "ensemble"
A bit of a hack, but it works.

In [ ]:
# For the all variants zero table

from typing import List, Dict, Tuple, Any
from fetch import _score_mapping

metrics: List[str] = ["disproportionality", "efficiency_gap", "geometric_seats_bias", "seats_bias", "votes_bias", "mean_median_average_district", "lopsided_outcomes", "declination"]
additions: Dict[str, str] = dict(zip(metrics, metrics))
_score_mapping.update(additions)


zero_table = {
    "FL_congress": {
        "disproportionality": 0.0000818182,
        "efficiency_gap": 0.0097272950,
        "geometric_seats_bias": 0.0912139487,
        "seats_bias": 0.0657093068,
        "votes_bias": 0.0648229398,
        "mean_median_average_district": 0.0557455566,
        "lopsided_outcomes": 0.1742642362,
        "declination": 0.0016454555,
    },
    "FL_upper": {
        "disproportionality": 0.0033136470,
        "efficiency_gap": 0.0802412077,
        "geometric_seats_bias": 0.4329517449,
        "seats_bias": 0.3582377983,
        "votes_bias": 0.3553741476,
        "mean_median_average_district": 0.1259505030,
        "lopsided_outcomes": 0.3804833128,
        "declination": 0.0156909732,
    },
    "FL_lower": {
        "disproportionality": 0.0000000000,
        "efficiency_gap": 0.0016545552,
        "geometric_seats_bias": 0.0262866005,
        "seats_bias": 0.0140410371,
        "votes_bias": 0.0137092146,
        "mean_median_average_district": 0.0006818216,
        "lopsided_outcomes": 0.0007818186,
        "declination": 0.0000000000,
    },
    "IL_congress": {
        "disproportionality": 1.0000000000,
        "efficiency_gap": 0.1786010298,
        "geometric_seats_bias": 0.0000090909,
        "seats_bias": 0.0004500048,
        "votes_bias": 0.0004454593,
        "mean_median_average_district": 0.0024091093,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0027318370,
    },
    "IL_upper": {
        "disproportionality": 1.0000000000,
        "efficiency_gap": 0.3916249631,
        "geometric_seats_bias": 0.0000000000,
        "seats_bias": 0.0000000000,
        "votes_bias": 0.0000000000,
        "mean_median_average_district": 0.0000000000,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0539139775,
    },
    "IL_lower": {
        "disproportionality": 1.0000000000,
        "efficiency_gap": 0.2688829873,
        "geometric_seats_bias": 0.0000000000,
        "seats_bias": 0.0000000000,
        "votes_bias": 0.0000000000,
        "mean_median_average_district": 0.0000000000,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0903502718,
    },
    "MI_congress": {
        "disproportionality": 0.0769777082,
        "efficiency_gap": 0.0188728434,
        "geometric_seats_bias": 0.0005681895,
        "seats_bias": 0.0008636457,
        "votes_bias": 0.0008636457,
        "mean_median_average_district": 0.0091000630,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0002772768,
    },
    "MI_upper": {
        "disproportionality": 0.0039681968,
        "efficiency_gap": 0.0000909091,
        "geometric_seats_bias": 0.0000000000,
        "seats_bias": 0.0000000000,
        "votes_bias": 0.0000000000,
        "mean_median_average_district": 0.0004863636,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0000000000,
    },
    "MI_lower": {
        "disproportionality": 0.0000000000,
        "efficiency_gap": 0.0000000000,
        "geometric_seats_bias": 0.0000000000,
        "seats_bias": 0.0000000000,
        "votes_bias": 0.0000000000,
        "mean_median_average_district": 0.0000000000,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0000000000,
    },
    "NC_congress": {
        "disproportionality": 0.0549412327,
        "efficiency_gap": 0.0806777434,
        "geometric_seats_bias": 0.1343507612,
        "seats_bias": 0.1329871169,
        "votes_bias": 0.1320143830,
        "mean_median_average_district": 0.2425921580,
        "lopsided_outcomes": 0.3407562639,
        "declination": 0.1364416798,
    },
    "NC_upper": {
        "disproportionality": 0.0033000041,
        "efficiency_gap": 0.0092318361,
        "geometric_seats_bias": 0.0114363898,
        "seats_bias": 0.0115136620,
        "votes_bias": 0.0113954800,
        "mean_median_average_district": 0.0812821105,
        "lopsided_outcomes": 0.0613774566,
        "declination": 0.0105182000,
    },
    "NC_lower": {
        "disproportionality": 0.0000000000,
        "efficiency_gap": 0.0000000000,
        "geometric_seats_bias": 0.0000000000,
        "seats_bias": 0.0000000000,
        "votes_bias": 0.0000000000,
        "mean_median_average_district": 0.0010591059,
        "lopsided_outcomes": 0.0000090909,
        "declination": 0.0000000000,
    },
    "NY_congress": {
        "disproportionality": 1.0000000000,
        "efficiency_gap": 1.0000000000,
        "geometric_seats_bias": 0.0001500000,
        "seats_bias": 0.0000363636,
        "votes_bias": 0.0000363636,
        "mean_median_average_district": 0.0000000000,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0000206962,
    },
    "NY_upper": {
        "disproportionality": 1.0000000000,
        "efficiency_gap": 0.9986863605,
        "geometric_seats_bias": 0.0000000000,
        "seats_bias": 0.0012590941,
        "votes_bias": 0.0012363668,
        "mean_median_average_district": 0.0000681818,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0004409177,
    },
    "NY_lower": {
        "disproportionality": 1.0000000000,
        "efficiency_gap": 0.2657603044,
        "geometric_seats_bias": 0.0000000000,
        "seats_bias": 0.0028636461,
        "votes_bias": 0.0027772820,
        "mean_median_average_district": 0.0000045455,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0000045455,
    },
    "OH_congress": {
        "disproportionality": 0.0002636366,
        "efficiency_gap": 0.0216637145,
        "geometric_seats_bias": 0.2873648030,
        "seats_bias": 0.2449237412,
        "votes_bias": 0.2433919166,
        "mean_median_average_district": 0.4740293374,
        "lopsided_outcomes": 0.9809771729,
        "declination": 0.2305054541,
    },
    "OH_upper": {
        "disproportionality": 0.0000000000,
        "efficiency_gap": 0.0007863689,
        "geometric_seats_bias": 0.0082228009,
        "seats_bias": 0.0111728309,
        "votes_bias": 0.0110682852,
        "mean_median_average_district": 0.1092597294,
        "lopsided_outcomes": 0.9048950498,
        "declination": 0.0074864184,
    },
    "OH_lower": {
        "disproportionality": 0.0000000000,
        "efficiency_gap": 0.0000000000,
        "geometric_seats_bias": 0.0000000000,
        "seats_bias": 0.0000000000,
        "votes_bias": 0.0000000000,
        "mean_median_average_district": 0.0002045470,
        "lopsided_outcomes": 0.3924975256,
        "declination": 0.0000000000,
    },
    "WI_congress": {
        "disproportionality": 0.0278819655,
        "efficiency_gap": 0.0172228150,
        "geometric_seats_bias": 0.0108636875,
        "seats_bias": 0.0111818718,
        "votes_bias": 0.0111318714,
        "mean_median_average_district": 0.0142546175,
        "lopsided_outcomes": 0.0001136366,
        "declination": 0.0040818395,
    },
    "WI_upper": {
        "disproportionality": 0.0000000000,
        "efficiency_gap": 0.0000000000,
        "geometric_seats_bias": 0.0000000000,
        "seats_bias": 0.0000000000,
        "votes_bias": 0.0000000000,
        "mean_median_average_district": 0.0000045455,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0000000000,
    },
    "WI_lower": {
        "disproportionality": 0.0000000000,
        "efficiency_gap": 0.0000000000,
        "geometric_seats_bias": 0.0000000000,
        "seats_bias": 0.0000000000,
        "votes_bias": 0.0000000000,
        "mean_median_average_district": 0.0000000000,
        "lopsided_outcomes": 0.0000000000,
        "declination": 0.0000000000,
    },
}

table = dict()
for combo, _values in zero_table.items():
    state, chamber = combo.split("_")
    table[(state, chamber)] = _values

latex_out: str = f'temp/partisan_zero_table.tex'
markdown_out: str = f'temp/partisan_zero_table.md'
make_table(table, latex_filename=latex_out, markdown_filename=markdown_out, values=False)

In [ ]:
# For the variants inverted zero tables

from typing import List, Dict, Tuple, Any
from fetch import _score_mapping

from rdapy import read_json

metrics: List[str] = ["disproportionality", "efficiency_gap", "geometric_seats_bias", "seats_bias", "votes_bias", "mean_median_average_district", "lopsided_outcomes", "declination"]
additions: Dict[str, str] = dict(zip(metrics, metrics))
_score_mapping.update(additions)

table_dir: str = (
    "~/Documents/work/Ensembles/partisan-bias-of-ensembles/tables/intermediate"
)

ensemble_filenames: List[str] = [
    "base0",  # Cut edges, minimum spanning tree
    "pop_minus",
    "pop_plus",
    "distpair",  # District pairs, minimum spanning tree
    "ust",  # Cut edges, uniform spanning tree
    "distpair_ust",  # District pairs, uniform spanning tree
    "reversible",  # The revised 1B sampled every 50K ensembles
    "county25",
    "county50",
    "county75",
    "county100",
]

for variant in ensemble_filenames:
    path = os.path.expanduser(f"{table_dir}/zero_table_{variant}-INVERTED.json")
    zero_table: Dict[str, Dict[str, float]] = read_json(path)

    table = dict()
    for combo, _values in zero_table.items():
        state, chamber = combo.split("_")
        table[(state, chamber)] = _values

    latex_out: str = f'temp/partisan_zero_table_{variant}-INVERTED.tex'
    markdown_out: str = f'temp/partisan_zero_table_{variant}-INVERTED.md'
    make_table(table, latex_filename=latex_out, markdown_filename=markdown_out, values=False)

In [ ]:
# For the all variants inverted zero table

from typing import List, Dict, Tuple, Any
from fetch import _score_mapping

from rdapy import read_json

metrics: List[str] = ["disproportionality", "efficiency_gap", "geometric_seats_bias", "seats_bias", "votes_bias", "mean_median_average_district", "lopsided_outcomes", "declination"]
additions: Dict[str, str] = dict(zip(metrics, metrics))
_score_mapping.update(additions)

table_dir: str = (
    "~/Documents/work/Ensembles/partisan-bias-of-ensembles/tables/intermediate"
)

path = os.path.expanduser(f"{table_dir}/zero_table_all-INVERTED.json")
zero_table: Dict[str, Dict[str, float]] = read_json(path)

table = dict()
for combo, _values in zero_table.items():
    state, chamber = combo.split("_")
    table[(state, chamber)] = _values

latex_out: str = f'temp/partisan_zero_table-INVERTED.tex'
markdown_out: str = f'temp/partisan_zero_table-INVERTED.md'
make_table(table, latex_filename=latex_out, markdown_filename=markdown_out, values=False)